# Homework 06: Data Preprocessing

This notebook applies reusable cleaning functions, compares the original and cleaned data, and saves a processed dataset.

In [1]:
# Packages used: numpy and pandas

## 1. Generate the Raw Dataset

The dataset contains missing numeric values and one mostly empty column so each cleaning decision is visible.

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

RAW = Path('data/raw')
PROCESSED = Path('data/processed')
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

raw_df = pd.DataFrame({
    'age': [34, 45, 29, 50, 38, np.nan, 41],
    'income': [55000, np.nan, 42000, 58000, np.nan, np.nan, 49000],
    'score': [0.82, 0.91, np.nan, 0.76, 0.88, 0.65, 0.79],
    'zipcode': ['90210', '10001', '60614', '94103', '73301', '12345', '94105'],
    'city': ['Beverly', 'New York', 'Chicago', 'SF', 'Austin', 'Unknown', 'San Francisco'],
    'extra_data': [np.nan, 42, np.nan, np.nan, np.nan, 5, np.nan],
})
raw_path = RAW / 'sample_data.csv'
raw_df.to_csv(raw_path, index=False)
print('Saved raw data:', raw_path)
raw_df

Saved raw data: data\raw\sample_data.csv


,age,income,score,zipcode,city,extra_data
0,34.0,55000.0,0.82,90210,Beverly,NaN
1,45.0,NaN,0.91,10001,New York,42.0
2,29.0,42000.0,NaN,60614,Chicago,NaN
3,50.0,58000.0,0.76,94103,SF,NaN
4,38.0,NaN,0.88,73301,Austin,NaN
5,NaN,NaN,0.65,12345,Unknown,5.0
6,41.0,49000.0,0.79,94105,San Francisco,NaN


## 2. Load and Inspect

In [3]:
df = pd.read_csv(raw_path, dtype={'zipcode': 'string'})
display(df.head())
pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum()})

,age,income,score,zipcode,city,extra_data
0,34.0,55000.0,0.82,90210,Beverly,NaN
1,45.0,NaN,0.91,10001,New York,42.0
2,29.0,42000.0,NaN,60614,Chicago,NaN
3,50.0,58000.0,0.76,94103,SF,NaN
4,38.0,NaN,0.88,73301,Austin,NaN


,dtype,missing
age,float64,1
income,float64,3
score,float64,1
zipcode,string,0
city,object,0
extra_data,float64,5


## 3. Apply the Cleaning Functions

`extra_data` is removed because more than half its values are missing. Numeric gaps are median-filled, then the modeling columns are standardized. Identifiers and labels are not normalized.

In [4]:
from src.cleaning import drop_missing, fill_missing_median, normalize_data

cleaned = drop_missing(df, threshold=0.50)
cleaned = fill_missing_median(cleaned, ['age', 'income', 'score'])
cleaned = normalize_data(cleaned, ['age', 'income', 'score'])
cleaned

,age,income,score,zipcode,city
0,-0.861209,0.767146,0.227593,90210,Beverly
1,0.861209,0.122743,1.374660,10001,New York
2,-1.644127,-2.025264,0.036415,60614,Chicago
3,1.644127,1.411548,-0.537119,94103,SF
4,-0.234875,0.122743,0.992304,73301,Austin
5,0.000000,0.122743,-1.939090,12345,Unknown
6,0.234875,-0.521659,-0.154763,94105,San Francisco


## 4. Compare Original and Cleaned Data

In [5]:
comparison = pd.DataFrame({
    'original_missing': df.isna().sum(),
    'cleaned_missing': cleaned.isna().sum(),
    'original_dtype': df.dtypes.astype(str),
    'cleaned_dtype': cleaned.dtypes.astype(str),
}).fillna('dropped')
display(comparison)

numeric_check = cleaned[['age', 'income', 'score']].agg(['mean', 'std']).round(4)
numeric_check

,original_missing,cleaned_missing,original_dtype,cleaned_dtype
age,1,0.0,float64,float64
city,0,0.0,object,object
extra_data,5,dropped,float64,dropped
income,3,0.0,float64,float64
score,1,0.0,float64,float64
zipcode,0,0.0,string,string


,age,income,score
mean,0.0000,0.0000,-0.0000
std,1.0801,1.0801,1.0801


## 5. Save the Processed Dataset

In [6]:
processed_path = PROCESSED / 'sample_data_cleaned.csv'
cleaned.to_csv(processed_path, index=False)
reloaded = pd.read_csv(processed_path, dtype={'zipcode': 'string'})
print('Saved processed data:', processed_path)
print('Shape:', reloaded.shape, '| Missing values:', int(reloaded.isna().sum().sum()))

Saved processed data: data\processed\sample_data_cleaned.csv
Shape: (7, 5) | Missing values: 0


## 6. Assumptions and Tradeoffs

- A median is appropriate for the small numeric sample because it is robust to extreme values, but it reduces natural variation.
- A column with more than 50% missingness is considered too weak for this exercise; a real project would first check whether its missingness carries information.
- Z-score scaling is useful for later models that compare coefficients or use distances. It is not applied to `zipcode`, which is an identifier rather than a quantity.
- Cleaning choices are documented because different assumptions can change later model results.